## Feature Engineering

This notebook contains feature engineering logic for WealthSync, focusing on two main categories:

1.  **Monthly Features (Models 1-4):** Derived from `data/processed/user_monthly.csv`.
2.  **Transaction Features (Model 5):** Derived from `data/processed/transactions_clean.csv`.

Every derived feature is a unit-tested, pure function of *past* data only (lags / running aggregates) to ensure no future information leaks into any row.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd

### Data Loading Helpers

These functions load our cleaned datasets from `/content/` so they are immediately available to our feature engineering and modeling stages.

In [2]:
from pathlib import Path
import pandas as pd

def get_data_filepath(filename: str) -> Path:
    """Finds the file under /content/ or falls back to the current directory."""
    primary_path = Path("/content/") / filename
    if primary_path.exists():
        return primary_path
    return Path(filename)

def load_transactions() -> pd.DataFrame:
    """Loads the transactions data with fallback routing."""
    path = get_data_filepath("transactions_clean.csv")
    return pd.read_csv(path, parse_dates=["date"])

def load_user_monthly() -> pd.DataFrame:
    """Loads the user monthly data with fallback routing."""
    path = get_data_filepath("user_monthly.csv")
    return pd.read_csv(path, parse_dates=["ym_dt"])

### Helper Functions

In [3]:
def safe_div(a: pd.Series, b: pd.Series, fill: float = 0.0) -> pd.Series:
    """Division that returns `fill` instead of inf/NaN (divide-by-zero guard)."""
    out = a.astype(float) / b.replace(0, np.nan).astype(float)
    return out.fillna(fill).replace([np.inf, -np.inf], fill)

In [4]:
def spending_stability(values: pd.Series, users: pd.Series, months: list[int]) -> pd.Series:
    """Calculates spending stability per user using only historical data.

    Stability is calculated as 1 / (1 + Coefficient of Variation) of past spending.
    """
    def _stability(s: pd.Series) -> pd.Series:
        # Vectorized shifting to get past months' values
        mat = np.column_stack([s.shift(m).values for m in months])
        mean = np.nanmean(mat, axis=1)
        std = np.nanstd(mat, axis=1)

        # Coefficient of variation (CV = std / mean), guarded against division by zero
        cv = np.where(mean > 1e-6, std / np.where(mean > 0, mean, 1.0), 0.0)
        return pd.Series(1.0 / (1.0 + cv), index=s.index)

    return values.groupby(users, sort=False).transform(_stability)

### Build Monthly Features (for Models 1-4)

This function constructs the feature table based on monthly user data, including direct copies, derived features, lagged values, rolling averages, and stability metrics.

In [5]:
def build_monthly_features(df: pd.DataFrame) -> pd.DataFrame:
    """Constructs the feature table for monthly models 1-4.

    Expects a DataFrame sorted by user_id and month (ym_dt).
    """
    base = df.sort_values(["user_id", "ym_dt"]).copy()

    f = pd.DataFrame(index=base.index)
    f["income"] = base["income"].astype(float).to_numpy()
    f["expense"] = base["expense"].astype(float).to_numpy()
    f["txn_count"] = base["txn_count"].to_numpy()
    f["avg_txn_amount"] = base["avg_txn_amount"].astype(float).to_numpy()

    g = base.groupby("user_id", sort=False)

    # Ratio and difference-based metrics
    f["cash_flow"] = f["income"] - f["expense"]
    f["expense_ratio"] = safe_div(f["expense"], f["income"])
    f["savings"] = f["cash_flow"]
    f["savings_ratio"] = safe_div(f["savings"], f["income"])
    f["rent_burden"] = safe_div(base["cat_spend_Rent"].astype(float), f["income"])

    for c in sorted(base.columns):
        if c.startswith("cat_spend_"):
            f[c.replace("cat_spend_", "cat_")] = base[c].astype(float)

    # 1-month and 2-month lags
    for m, name in ((1, "income_1m"), (2, "income_2m"), (1, "expense_1m"), (2, "expense_2m")):
        f[name] = g["income" if name.startswith("income") else "expense"].shift(m)
    f["cash_flow_1m"] = f["cash_flow"].groupby(base["user_id"], sort=False).shift(1)
    f["expense_ratio_1m"] = f["expense_ratio"].groupby(base["user_id"], sort=False).shift(1)
    f["savings_1m"] = f["savings"].groupby(base["user_id"], sort=False).shift(1)

    # Rolling 3-month windows (prior months only)
    f["expense_3m_mean"] = f["expense"].groupby(base["user_id"], sort=False).transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    f["income_3m_mean"] = f["income"].groupby(base["user_id"], sort=False).transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    f["cash_flow_3m_mean"] = f["cash_flow"].groupby(base["user_id"], sort=False).transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    f["savings_3m_mean"] = f["savings"].groupby(base["user_id"], sort=False).transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

    # Stability and Trend descriptors
    f["income_trend"] = f["income_1m"] - f["income_2m"]
    f["expense_trend"] = f["expense_1m"] - f["expense_2m"]
    f["spending_stability"] = spending_stability(f["expense"], base["user_id"], [1, 2, 3])
    f["income_stability"] = spending_stability(f["income"], base["user_id"], [1, 2, 3])

    # Re-attach identifying columns
    f["user_id"] = base["user_id"].values
    f["ym_dt"] = base["ym_dt"].values
    f["month"] = base["month"].values
    f["year"] = base["year"].values
    return f

### Build Transaction Features (for Model 5)

This function creates a feature table for Model 5, where each row represents an expense transaction. Features include transaction amount, category, timing, running monthly income/expense before the transaction, and historical payment method counts.

In [6]:
def build_transaction_features(txn: pd.DataFrame) -> pd.DataFrame:
    """Generates historical features for Model 5 (expense transactions).

    Ensures only past/current-time information is used to avoid leakage.
    """
    txn = txn.sort_values(["user_id", "date"]).copy()
    txn = txn[txn["transaction_type"] == "expense"].copy()
    txn = txn.reset_index(drop=True)

    f = pd.DataFrame(index=txn.index)
    f["amount"] = txn["amount"].astype(float)
    f["month"] = txn["date"].dt.month
    f["day_of_month"] = txn["date"].dt.day
    f["is_weekend"] = txn["date"].dt.weekday.isin([5, 6]).astype(int)

    # Cumulative monthly cashflow and income up to the current transaction
    amt = (txn["amount"].astype(float) * (txn["transaction_type"] == "income")) - (txn["amount"].astype(float) * (txn["transaction_type"] == "expense"))
    f["month_cash_flow_so_far"] = amt.groupby([txn["user_id"], txn["ym"]]).cumsum() - amt

    income_amt = txn["amount"].astype(float) * (txn["transaction_type"] == "income")
    f["month_income_so_far"] = income_amt.groupby([txn["user_id"], txn["ym"]]).cumsum() - income_amt

    # Payment method historical count before the current transaction
    for method in ["UPI", "Debit Card", "Cash", "Bank Transfer"]:
        ind = (txn["payment_mode"] == method).astype(int)
        f[f"hist_{method.replace(' ', '_')}"] = ind.groupby(txn["user_id"]).cumsum() - ind
    f["hist_total"] = txn.groupby("user_id").cumcount()

    # Category-wise expenditure prior to the current transaction
    cat_labels = txn["category"].astype(str).str.replace(" ", "_")
    wide = (txn[["user_id", "amount"]]
            .assign(_cat=cat_labels)
            .pivot_table(index=txn.index, columns="_cat", values="amount", aggfunc="sum", fill_value=0.0))
    wide = wide.reindex(columns=sorted(cat_labels.unique()), fill_value=0.0)
    cat_prev = wide.groupby(txn["user_id"], sort=False).cumsum() - wide
    for c in sorted(cat_labels.unique()):
        f[f"cat_cum_{c}"] = cat_prev[c].to_numpy()

    f["user_id"] = txn["user_id"].values
    f["date"] = txn["date"].values
    f["ym"] = txn["ym"].values
    f["category"] = txn["category"].values
    f["payment_mode"] = txn["payment_mode"].values
    return f

## WealthSync - Target / Label Generation (Models 1-5)

This section outlines the generation of targets and labels for various WealthSync models. It's crucial to understand the nature of these targets:

*   **Forecasting Targets (Models 1 & 2):** These are actual future observed values (e.g., next month's expense or savings). They are *not* rule-derived.
*   **Rule-Derived Labels (Models 3, 4, 5):** These are generated from deterministic, documented scoring rules. They *must never* be presented as ground truth, as they are based on definitions within the system rather than external verified data.

In [7]:
from __future__ import annotations

import numpy as np
import pandas as pd

### Model 1 & 2 - Future Targets

This function adds future expense and savings targets for models 1 and 2, which are forecasting models. These targets represent the actual observed values in the subsequent month.

In [8]:
def add_future_targets(features: pd.DataFrame) -> pd.DataFrame:
    """Generates prediction targets for Models 1 & 2 by shifting values by -1.

    * future_expense (t+1)
    * future_savings (t+1)
    """
    df = features.sort_values(["user_id", "ym_dt"]).copy()
    g = df.groupby("user_id", sort=False)

    df["future_expense"] = g["expense"].shift(-1)
    df["future_savings"] = g["savings"].shift(-1)
    return df

### Model 3 - Financial Wellness Score (Rule-Derived)

This function calculates a Financial Wellness Score based on a set of predefined rules. The score ranges from 0 to 100, starting at 50, and adjusts based on various financial metrics like expense ratio, savings ratio, cash flow, rent burden, and stability of income and spending.

In [9]:
def wellness_score(df: pd.DataFrame) -> pd.Series:
    # Financial Wellness Score generator [0, 100] (Rule-Derived Target). Scores households based on expense ratios, savings ratios, cash flow, rent burden, and income/expense stability.

    def rng_bonus(series, lo, hi, amount):
        return np.where(series.eq(0) | series.isna(), 0.0,
                        np.where((series >= lo) & (series < hi), amount, 0.0))

    score = np.full(len(df), 50.0)

    # 1. Expense Ratio Evaluator
    er = df["expense_ratio"].replace(0, np.nan)
    score += rng_bonus(er, 0.0, 0.50, 20.0)
    score += rng_bonus(er, 0.50, 0.70, 12.0)
    score += rng_bonus(er, 0.70, 0.90, 5.0)
    score += np.where(er > 0.90, -15.0, 0.0)

    # 2. Savings Ratio Evaluator
    sr = df["savings_ratio"].replace(0, np.nan)
    score += rng_bonus(sr, 0.20, np.inf, 15.0)
    score += rng_bonus(sr, 0.10, 0.20, 8.0)
    score += np.where(sr < 0, -15.0, 0.0)

    # 3. Cash Flow Evaluator
    cf = df["cash_flow"].fillna(0)
    score += np.where(cf > 0, 10.0, -10.0)

    # 4. Rent Burden Evaluator
    rb = df["rent_burden"].replace(0, np.nan)
    score += np.where((rb > 0) & (rb < 0.30), 5.0, 0.0)
    score += np.where(rb > 0.50, -5.0, 0.0)

    # 5. Volatility / Stability Evaluator
    st = df["spending_stability"].fillna(0.5)
    sti = df["income_stability"].fillna(0.5)
    score += np.where(st >= 0.70, 5.0, np.where(st <= 0.35, -5.0, 0.0))
    score += np.where(sti >= 0.70, 5.0, np.where(sti <= 0.35, -5.0, 0.0))

    return pd.Series(np.clip(score, 0, 100).round(0).astype(int), index=df.index)

### Model 4 - Financial Risk (Rule-Derived)

This function identifies a binary Financial Risk label (1 = HIGH RISK, 0 = LOW RISK) based on several conditions related to cash flow, expense ratio, savings ratio, and volatility of income and spending over the trailing 6 months.

In [10]:
def risk_flag(df: pd.DataFrame) -> pd.Series:
    """Computes binary Financial Risk labels [0, 1] (Rule-Derived Target).

    Flags risk based on persistent negative cashflow, critical expense levels,
    and historical volatility.
    """
    uid = df["user_id"]
    val = pd.DataFrame(index=df.index)

    cash_flow = df["cash_flow"]
    expense = df["expense"]
    income = df["income"]

    for lag in range(6):
        val[f"cf_{lag}"] = cash_flow.groupby(uid, sort=False).shift(lag)
        val[f"exp_{lag}"] = expense.groupby(uid, sort=False).shift(lag)
        val[f"inc_{lag}"] = income.groupby(uid, sort=False).shift(lag)

    cf6 = val[[f"cf_{i}" for i in range(6)]].fillna(0)
    exp6 = val[[f"exp_{i}" for i in range(6)]]
    inc6 = val[[f"inc_{i}" for i in range(6)]]

    # Apply rules
    r1 = (cf6 < 0).sum(axis=1) >= 3
    r2 = (df["expense_ratio"] > 1.0) & (df["savings_ratio"] < 0)

    inc_mean = inc6.mean(axis=1).replace(0, np.nan)
    inc_cv = inc6.std(axis=1) / inc_mean
    r3 = (inc_cv.fillna(0) > 0.60) & (df["cash_flow"] < 0)

    exp_mean = exp6.mean(axis=1).replace(0, np.nan)
    exp_cv = exp6.std(axis=1) / exp_mean
    r4 = (exp_cv.fillna(0) > 1.0) & (df["cash_flow"] < 0)

    return pd.Series((r1 | r2 | r3 | r4).astype(int), index=df.index)

### Model 5 - Best Payment Method (Observed-Behaviour Target)

For Model 5, the 'best payment method' target is simply the payment mode *actually used* for each transaction. This is an observed-behaviour label, meaning the model learns from user choices rather than an external 'ground truth' definition of best.

In [11]:
PAYMENT_METHODS = ["UPI", "Debit Card", "Cash", "Bank Transfer"]

def payment_method_target(txn_features: pd.DataFrame) -> pd.Series:
    """Best-payment-method target for Model 5.

    The dataset has no independent ground truth for "best". The target is the
    payment mode *actually used* on each transaction. Interpretation
    (documented): a rational user picks a method based on transaction size,
    category, and available cash flow; the model learns that suitability
    function. This is an observed-behaviour label, not verified ground truth.
    """
    # The target is directly the 'payment_mode' column from the transaction features.
    return txn_features["payment_mode"].astype(str)

In [12]:
from __future__ import annotations

# ---------------------------------------------------------
# Import required libraries
# ---------------------------------------------------------

import itertools
import json
from datetime import date
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

# Import evaluation metrics for regression and classification
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)

# Pipeline and scaling utilities
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# Configure Artifact Output Directory
# ---------------------------------------------------------

# All trained models, metrics, feature lists, and metadata
# will be saved inside this folder in Google Colab.
ARTIFACTS_DIR = Path("/content/artifacts")

# Create the folder if it does not already exist
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Time Split Configuration
# ---------------------------------------------------------

# Training period
TRAIN_MONTHS = ("2021-01", "2023-06")

# Validation period
VAL_MONTHS = ("2023-07", "2023-12")

# Testing period
TEST_MONTHS = ("2024-01", "2024-12")


# ---------------------------------------------------------
# Split Monthly Data
# ---------------------------------------------------------

def split_by_ym(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    # Convert the ym_dt column into YYYY-MM format
    ym = df["ym_dt"].dt.strftime("%Y-%m")

    # Helper function to select a specific time window
    def window(lo: str, hi: str) -> pd.Series:
        return (ym >= lo) & (ym <= hi)

    # Return training, validation, and test datasets
    return (
        df[window(*TRAIN_MONTHS)],
        df[window(*VAL_MONTHS)],
        df[window(*TEST_MONTHS)]
    )


# ---------------------------------------------------------
# Split Transaction Data Based on Date
# ---------------------------------------------------------

def split_transactions_by_date(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    # Convert transaction date column into datetime format
    d = pd.to_datetime(df["date"])

    # Helper function to select a date range
    def window(lo: str, hi: str) -> pd.Series:
        return (
            (d >= pd.Timestamp(lo)) &
            (d <= pd.Timestamp(hi))
        )

    # Return training, validation, and test transactions
    return (
        df[window("2021-01-01", "2023-06-30")],
        df[window("2023-07-01", "2023-12-31")],
        df[window("2024-01-01", "2024-12-31")]
    )


# ---------------------------------------------------------
# Regression Evaluation Metrics
# ---------------------------------------------------------

def regression_metrics(y_true, y_pred) -> dict[str, float]:

    # Calculate Mean Squared Error
    mse = mean_squared_error(y_true, y_pred)

    # Return commonly used regression metrics
    return {
        # MAE = average absolute prediction error
        "MAE": round(
            float(mean_absolute_error(y_true, y_pred)), 3
        ),

        # RMSE = square root of mean squared error
        "RMSE": round(
            float(np.sqrt(mse)), 3
        ),

        # R2 = how well the model explains variation in target
        "R2": round(
            float(r2_score(y_true, y_pred)), 4
        ),
    }


# ---------------------------------------------------------
# Classification Evaluation Metrics
# ---------------------------------------------------------

def classification_metrics(
    y_true,
    y_pred,
    y_prob=None
) -> dict[str, float]:

    # Count number of unique classes
    n_classes = len(set(y_true))

    # Calculate basic classification metrics
    metrics = {

        # Percentage of correct predictions
        "Accuracy": round(
            float(accuracy_score(y_true, y_pred)), 4
        ),

        # Weighted precision
        "Precision_weighted": round(
            float(
                precision_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            ),
            4
        ),

        # Weighted recall
        "Recall_weighted": round(
            float(
                recall_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            ),
            4
        ),

        # Weighted F1 score
        "F1_weighted": round(
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            ),
            4
        ),
    }


    # -----------------------------------------------------
    # ROC-AUC
    # -----------------------------------------------------

    # ROC-AUC is calculated only when:
    # 1. There are exactly two classes
    # 2. Probability predictions are available
    if n_classes == 2 and y_prob is not None:

        try:
            metrics["ROC_AUC"] = round(
                float(
                    roc_auc_score(
                        y_true,
                        y_prob
                    )
                ),
                4
            )

        # If ROC-AUC cannot be calculated, skip it
        except ValueError:
            pass


    # -----------------------------------------------------
    # Confusion Matrix
    # -----------------------------------------------------

    try:
        metrics["Confusion_Matrix"] = (
            confusion_matrix(
                y_true,
                y_pred
            ).tolist()
        )

    except Exception:
        pass


    # Return all calculated metrics
    return metrics


# ---------------------------------------------------------
# Hyperparameter Search
# ---------------------------------------------------------

def search_params(
    estimator,
    param_grid: dict[str, list],
    X: pd.DataFrame,
    y,
    X_val,
    y_val,
    metric_fn,
    maximize=True,
    predict_fn=None
):

    """
    Performs validation-set grid search.

    It tries different combinations of hyperparameters,
    trains a model for each combination, evaluates it on
    validation data, and returns the best model.
    """

    # By default, use model.predict() for predictions
    if predict_fn is None:
        predict_fn = lambda m, X: m.predict(X)

    # Get hyperparameter names
    keys = list(param_grid)

    # Store the best score and best model
    best_score, best = None, None


    # Generate every possible combination of parameters
    for combo in itertools.product(*param_grid.values()):

        # Convert parameter values into a dictionary
        params = dict(zip(keys, combo))

        # Create model using current parameters
        model = estimator(**params)

        # Train the model on training data
        model.fit(X, y)

        # Make predictions on validation data
        pred = predict_fn(model, X_val)

        # Calculate validation metric
        score = metric_fn(y_val, pred)

        # Check whether this model is better than the previous best model
        if (
            best_score is None
            or (
                score > best_score
                if maximize
                else score < best_score
            )
        ):
            best_score = score

            # Store best model and parameters
            best = (model, params)


    # Return:
    # 1. Best model
    # 2. Best parameters
    # 3. Best validation score
    return best[0], best[1], best_score


# ---------------------------------------------------------
# Save Model and Prediction Metadata
# ---------------------------------------------------------

def save_predict_api_meta(
    name: str,
    model: Any,
    features: list[str],
    art_dir: Path,
    metrics: dict,
    algorithm: str,
    version: str,
    target: str,
    target_desc: str,
    scaler=None,
    label_mapping: dict | None = None,
    extra: dict | None = None
):

    """
    Saves the trained model and all related metadata.
    """

    # Create artifact directory
    art_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # -----------------------------------------------------
    # Save trained model
    # -----------------------------------------------------

    joblib.dump(
        model,
        art_dir / "model.joblib"
    )


    # Save scaler if one was used
    if scaler is not None:
        joblib.dump(
            scaler,
            art_dir / "scaler.pkl"
        )


    # -----------------------------------------------------
    # Save feature names
    # -----------------------------------------------------

    (art_dir / "feature_columns.json").write_text(
        json.dumps(
            features,
            indent=2
        ),
        encoding="utf-8"
    )


    # -----------------------------------------------------
    # Save training metrics
    # -----------------------------------------------------

    metrics_path = (
        art_dir / "training_metrics.json"
    )

    metrics_path.write_text(
        json.dumps(
            metrics,
            indent=2,
            default=str
        ),
        encoding="utf-8"
    )


    # -----------------------------------------------------
    # Create model metadata
    # -----------------------------------------------------

    meta = {

        # Name of the model
        "model_name": name,

        # Algorithm used
        "algorithm": algorithm,

        # Model version
        "version": version,

        # Features used by the model
        "features": features,

        # Target variable
        "target": target,

        # Description of target variable
        "target_description": target_desc,

        # Date on which model was trained
        "training_date": date.today().isoformat(),

        # Model performance metrics
        "metrics": metrics,

        # Mapping for classification labels if applicable
        "label_mapping": label_mapping,

        # Additional notes
        "notes": extra,
    }


    # Save model metadata
    (
        art_dir / "model_metadata.json"
    ).write_text(
        json.dumps(
            meta,
            indent=2,
            default=str
        ),
        encoding="utf-8"
    )


    # -----------------------------------------------------
    # Save label mapping if available
    # -----------------------------------------------------

    if label_mapping is not None:

        (
            art_dir / "label_mapping.json"
        ).write_text(
            json.dumps(
                label_mapping,
                indent=2
            ),
            encoding="utf-8"
        )


    # Print where the artifacts were saved
    print(
        f"Artifacts saved under -> {art_dir}"
    )


# ---------------------------------------------------------
# Fit Feature Scaler
# ---------------------------------------------------------

def get_scaler_fitted(
    X_train: pd.DataFrame
) -> Pipeline:

    """
    Creates and fits a StandardScaler.

    Important:
    The scaler is fitted ONLY on training data to
    prevent data leakage.
    """

    return Pipeline(
        [
            ("scale", StandardScaler())
        ]
    ).fit(X_train)


# ---------------------------------------------------------
# Log Transformation
# ---------------------------------------------------------

def signed_log1p(y) -> np.ndarray:

    # Convert input into NumPy array of float values
    y = np.asarray(
        y,
        dtype=float
    )

    # Apply signed log transformation
    #
    # log1p(x) = log(1 + x)
    #
    # Sign is preserved for negative values.
    return (
        np.sign(y)
        * np.log1p(np.abs(y))
    )


# ---------------------------------------------------------
# Reverse Log Transformation
# ---------------------------------------------------------

def signed_expm1(y) -> np.ndarray:

    # Reverse the signed_log1p transformation
    return (
        np.sign(y)
        * np.expm1(np.abs(y))
    )


# ---------------------------------------------------------
# Make Predictions Using a Pipeline
# ---------------------------------------------------------

def scaled_predict(
    pipe: Pipeline,
    X: pd.DataFrame
):

    # Use the trained pipeline to make predictions
    return pipe.predict(X)

## Deep Learning Models for Tabular Data

We add two PyTorch-based Deep Learning models to compare with the traditional ML models:

1. **MLP (Multi-Layer Perceptron)** - Simple fully-connected feed-forward network
2. **ResNet-style MLP** - MLP with residual connections (skip connections), effective for tabular data (see *TabResNet* / *ResNet for Tabular Data* literature)

**Why NOT LSTM?** The monthly features are *already aggregated* per user-month (income, expense, ratios, lags, rolling means). Each row is an independent sample with engineered temporal features (lags, trends, stability). There is no raw sequential input (like daily transactions) to feed an LSTM. Reshaping tabular rows into fake sequences would introduce leakage and add no signal. The ResNet-style MLP is the appropriate second DL architecture for this tabular setting.

Both models use:
- Standardized inputs (StandardScaler fitted on TRAIN only)
- Early stopping on validation loss
- Reproducible seeds
- Lightweight architectures for laptop training
- Regression head (MSE loss) for spending/savings/wellness; Classification head (BCEWithLogitsLoss) for risk

In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import copy

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ---------------------------------------------------------
# Model 1: Simple MLP
# ---------------------------------------------------------
class MLPRegressor(nn.Module):
    """Simple MLP for regression (spending, savings, wellness)."""
    def __init__(self, input_dim: int, hidden_dims=(128, 64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))  # regression output
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPClassifier(nn.Module):
    """Simple MLP for binary classification (risk)."""
    def __init__(self, input_dim: int, hidden_dims=(128, 64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))  # logit output
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ---------------------------------------------------------
# Model 2: ResNet-style MLP (Tabular ResNet)
# ---------------------------------------------------------
class ResidualBlock(nn.Module):
    """Residual block with skip connection."""
    def __init__(self, dim: int, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))


class ResNetMLPRegressor(nn.Module):
    """ResNet-style MLP for regression."""
    def __init__(self, input_dim: int, hidden_dim=128, n_blocks=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.blocks = nn.Sequential(*[ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)])
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.blocks(x)
        return self.output(x).squeeze(-1)


class ResNetMLPClassifier(nn.Module):
    """ResNet-style MLP for binary classification."""
    def __init__(self, input_dim: int, hidden_dim=128, n_blocks=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.blocks = nn.Sequential(*[ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)])
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.blocks(x)
        return self.output(x).squeeze(-1)


# ---------------------------------------------------------
# Training Utilities
# ---------------------------------------------------------
def train_dl_model(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    task: str = 'regression',  # 'regression' or 'classification'
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    patience: int = 15,
    verbose: bool = True
) -> tuple[nn.Module, dict]:
    """
    Train a PyTorch model with early stopping on validation loss.
    Returns (best_model, history_dict).
    """
    model = model.to(DEVICE)
    
    # DataLoaders
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    val_ds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    # Loss & Optimizer
    if task == 'regression':
        criterion = nn.MSELoss()
    else:
        criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, verbose=False)

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # Training
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_losses.append(loss.item())

        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)

        scheduler.step(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1}')
                break

        if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
            print(f'  Epoch {epoch+1:3d} | Train: {avg_train:.4f} | Val: {avg_val:.4f}')

    # Load best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


def evaluate_dl_regression(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_test).to(DEVICE)
        pred = model(X_tensor).cpu().numpy()
    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)
    return {'MAE': round(float(mae), 3), 'MSE': round(float(mse), 3), 'RMSE': round(float(rmse), 3), 'R2': round(float(r2), 4)}, pred


def evaluate_dl_classification(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_test).to(DEVICE)
        logits = model(X_tensor).cpu().numpy()
        probs = 1 / (1 + np.exp(-logits))  # sigmoid
        preds = (probs >= 0.5).astype(int)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(y_test, probs)
    except ValueError:
        auc = None
    return {'Accuracy': round(float(acc), 4), 'Precision_weighted': round(float(prec), 4), 'Recall_weighted': round(float(rec), 4), 'F1_weighted': round(float(f1), 4), 'ROC_AUC': round(float(auc), 4) if auc else None}, preds, probs

**Train spending model**

In [13]:
from __future__ import annotations

import numpy as np
import pandas as pd

# Import regression algorithms
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor


# Name of this ML training run
RUN_NAME = "spending"

# Features that will be used by the ML models
FEATURES = [
    "income", "expense", "cash_flow", "expense_ratio",
    "expense_1m", "expense_2m", "expense_3m_mean",
    "income_1m", "income_2m", "income_3m_mean",
    "cash_flow_1m", "cash_flow_3m_mean",
    "expense_trend", "income_trend",
    "spending_stability", "income_stability",
    "txn_count", "avg_txn_amount", "month", "year",
]

# Different regression algorithms that we want to compare
CANDIDATES = {
    "RandomForestRegressor": RandomForestRegressor,
    "GradientBoostingRegressor": GradientBoostingRegressor,
    "XGBRegressor": XGBRegressor,
}

# Hyperparameter values that will be tested for each algorithm
# The best combination will be selected using the validation set
GRID = {
    "RandomForestRegressor": {
        "n_estimators": [100, 250],
        "max_depth": [None, 10],
        "min_samples_leaf": [1, 4],
    },

    "GradientBoostingRegressor": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
    },

    "XGBRegressor": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
        "subsample": [0.8, 1.0],
    },
}


# Build the final training dataset
def build_training_frame() -> pd.DataFrame:

    # Load monthly user-level financial data
    monthly = load_user_monthly()

    # Create different features from the monthly data
    feats = build_monthly_features(monthly)

    # Create the target variable:
    # future_expense = spending in the future month
    feats = add_future_targets(feats)

    # Combine all category spending columns into one feature
    # cat_ columns represent spending for different categories
    feats["category_spend"] = feats[
        [c for c in feats.columns if c.startswith("cat_")]
    ].sum(axis=1)

    # Remove rows where required features or target are missing
    return feats.dropna(
        subset=FEATURES + ["future_expense"]
    )


# Main training pipeline
def main() -> None:

    # Build the complete dataset containing features and target
    df = build_training_frame()

    # Find all category-based features
    # Example: cat_food, cat_transport, cat_shopping, etc.
    cat_cols = [
        c for c in df.columns
        if c.startswith("cat_")
    ]

    # Keep only category columns that actually contain some data
    cat_used = [
        c for c in cat_cols
        if df[c].sum() > 0
    ]

    # Combine normal features + useful category features
    all_features = FEATURES + cat_used


    # Split data into:
    # Training set
    # Validation set
    # Test set

    # split_by_ym() likely splits the data based on year/month so that future data is not used for training.
    train, val, test = split_by_ym(df)


    # Prepare training data

    # Xtr = training input features
    # ytr = training target

    # signed_log1p() transforms the target to reduce the effect of very large spending values.
    Xtr = train[all_features]
    ytr = signed_log1p(train["future_expense"])


    # Validation data
    Xva = val[all_features]
    yva = signed_log1p(val["future_expense"])


    # Test data
    Xte = test[all_features]

    # Keep original test target values for final evaluation
    yte_orig = test["future_expense"].astype(float)


    # Dictionary to store the results of every algorithm
    results = {}


    # Train and compare all candidate algorithms
    for name, est in CANDIDATES.items():

        # Search through different hyperparameter combinations and find the best parameters for this algorithm.

        # MAE (Mean Absolute Error) is used as the evaluation metric. Lower MAE is better.
        model, params, mae = search_params(
            est,
            GRID[name],
            Xtr,
            ytr,
            Xva,
            yva,

            # Calculate MAE for validation predictions
            metric_fn=lambda yt, yp:
                regression_metrics(yt, yp)["MAE"],

            # We want to MINIMIZE MAE
            maximize=False
        )

        # Store validation MAE and best parameters
        results[name] = {
            "val_MAE_log": mae,
            "params": params
        }


    # Select the best algorithm : Find the algorithm having the lowest validation MAE
    best_name = min(
        results,
        key=lambda k: results[k]["val_MAE_log"]
    )

    # Get the best hyperparameters of the selected algorithm
    params = results[best_name]["params"]


    # -----------------------------------------------------
    # Final model training
    # -----------------------------------------------------

    # Create a new model using the best algorithm and its best hyperparameters
    model = CANDIDATES[best_name](**params)


    # Train the final model using BOTH : Training data + Validation data
    # At this point, we already used the validation set to select the best algorithm and parameters. Therefore, we can now use it for final training.
    model.fit(
        pd.concat([Xtr, Xva], axis=0),
        np.concatenate([ytr, yva])
    )


    # -----------------------------------------------------
    # Evaluate the final model on TEST data
    # -----------------------------------------------------

    # Make predictions on completely unseen test data
    test_predictions = signed_expm1(
        model.predict(Xte)
    )

    # Calculate final test metrics
    test_metrics = regression_metrics(
        yte_orig,
        test_predictions
    )


    # -----------------------------------------------------
    # Evaluate the final model on VALIDATION data
    # -----------------------------------------------------

    # Generate predictions for validation data
    val_predictions = signed_expm1(
        model.predict(Xva)
    )

    # Calculate validation metrics
    val_metrics = regression_metrics(
        signed_expm1(yva),
        val_predictions
    )


    # -----------------------------------------------------
    # Create a comparison of all algorithms
    # -----------------------------------------------------

    # Store the validation MAE and final parameters
    # for every algorithm that was tested.
    comparison = {
        k: {
            "val_MAE_log": v["val_MAE_log"],
            "final_params": v["params"]
        }
        for k, v in results.items()
    }


    # -----------------------------------------------------
    # Save the trained model and its metadata
    # -----------------------------------------------------



# ---------------------------------------------------------
# Deep Learning Models (Regression)
# ---------------------------------------------------------

# Scale features for DL (StandardScaler fitted on TRAIN only)
scaler_dl = StandardScaler().fit(Xtr)
Xtr_dl = scaler_dl.transform(Xtr)
Xva_dl = scaler_dl.transform(Xva)
Xte_dl = scaler_dl.transform(Xte)

# Prepare y arrays for DL
ytr = signed_log1p(train["future_expense"])
yva = signed_log1p(val["future_expense"])
yte_orig = test["future_expense"].astype(float)

# Convert to numpy
ytr_np = ytr.values if hasattr(ytr, 'values') else np.array(ytr)
yva_np = yva.values if hasattr(yva, 'values') else np.array(yva)

input_dim = Xtr_dl.shape[1]

print(f"\n=== Training Deep Learning Models for Spending Prediction ===")
print(f"Input dim: {input_dim} | Train: {len(Xtr_dl)} | Val: {len(Xva_dl)} | Test: {len(Xte_dl)}")

# ---- MLP ----
print("\n--- MLP ---")
mlp_reg = MLPRegressor(input_dim, hidden_dims=(128, 64, 32), dropout=0.2)
mlp_reg, mlp_history = train_dl_model(
    mlp_reg, Xtr_dl, ytr_np, Xva_dl, yva_np,
    task='regression', epochs=100, batch_size=256, lr=1e-3, patience=15
)
mlp_metrics, mlp_preds = evaluate_dl_regression(mlp_reg, Xte_dl, yte_orig)
print(f"MLP Test: MAE={mlp_metrics['MAE']:.2f}, RMSE={mlp_metrics['RMSE']:.2f}, R2={mlp_metrics['R2']:.4f}")

# ---- ResNet-style MLP ----
print("\n--- ResNet MLP ---")
resnet_reg = ResNetMLPRegressor(input_dim, hidden_dim=128, n_blocks=3, dropout=0.2)
resnet_reg, resnet_history = train_dl_model(
    resnet_reg, Xtr_dl, ytr_np, Xva_dl, yva_np,
    task='regression', epochs=100, batch_size=256, lr=1e-3, patience=15
)
resnet_metrics, resnet_preds = evaluate_dl_regression(resnet_reg, Xte_dl, yte_orig)
print(f"ResNet MLP Test: MAE={resnet_metrics['MAE']:.2f}, RMSE={resnet_metrics['RMSE']:.2f}, R2={resnet_metrics['R2']:.4f}")

# ---------------------------------------------------------
# Combine DL results with traditional ML comparison
# ---------------------------------------------------------

# Add DL models to comparison
comparison["MLP"] = {"test_MAE": mlp_metrics['MAE'], "test_RMSE": mlp_metrics['RMSE'], "test_R2": mlp_metrics['R2'], "params": "hidden=(128,64,32), dropout=0.2"}
comparison["ResNet_MLP"] = {"test_MAE": resnet_metrics['MAE'], "test_RMSE": resnet_metrics['RMSE'], "test_R2": resnet_metrics['R2'], "params": "hidden=128, blocks=3, dropout=0.2"}

# Save DL model artifacts
import joblib
joblib.dump(mlp_reg.state_dict(), ARTIFACTS_DIR / "spending" / "mlp_state_dict.pt")
joblib.dump(scaler_dl, ARTIFACTS_DIR / "spending" / "scaler_dl.pkl")
joblib.dump(resnet_reg.state_dict(), ARTIFACTS_DIR / "spending" / "resnet_state_dict.pt")

# Update metrics dict for saving
metrics["test_dl"] = {
    "MLP": mlp_metrics,
    "ResNet_MLP": resnet_metrics
}
metrics["algorithm_comparison"] = comparison


    save_predict_api_meta(
        name=RUN_NAME,

        # Save the final trained model
        model=model,

        # Save the list of features used by the model
        features=all_features,

        # Folder where model artifacts will be saved
        art_dir=ARTIFACTS_DIR / RUN_NAME,

        # Save validation metrics, test metrics, and algorithm comparison results
        metrics={
            "val": val_metrics,
            "test": test_metrics,
            "algorithm_comparison": comparison
        },

        # Save the name of the selected algorithm
        algorithm=best_name,

        # Model version
        version="1.0",

        # Name of the target variable
        target="future_expense",

        # Human-readable description of the target
        target_desc="Predicted future-month aggregate spending.",
    )

if __name__ == "__main__":
    main()

/tmp/ipykernel_7587/3169358447.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(mat, axis=1)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_7587/3169358447.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(mat, axis=1)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Artifacts saved under -> /content/artifacts/spending


**Train Saving model**

In [14]:
from __future__ import annotations

# ---------------------------------------------------------
# Import required libraries
# ---------------------------------------------------------

import numpy as np
import pandas as pd

# Import regression algorithms
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)

from xgboost import XGBRegressor


# ---------------------------------------------------------
# Model / Training Run Configuration
# ---------------------------------------------------------

# Name of this ML model/training run
RUN_NAME = "savings"


# ---------------------------------------------------------
# Features used for predicting future savings
# ---------------------------------------------------------

FEATURES = [
    "income",
    "expense",
    "cash_flow",
    "expense_ratio",
    "savings_ratio",

    # Historical savings features
    "savings_1m",
    "savings_3m_mean",

    # Historical income features
    "income_1m",
    "income_2m",
    "income_3m_mean",

    # Historical expense features
    "expense_1m",
    "expense_2m",
    "expense_3m_mean",

    # Historical cash-flow features
    "cash_flow_1m",
    "cash_flow_3m_mean",

    # Trend-based features
    "income_trend",
    "expense_trend",

    # Stability features
    "spending_stability",
    "income_stability",

    # Transaction-level features
    "txn_count",
    "avg_txn_amount",

    # Time features
    "month",
    "year",
]


# ---------------------------------------------------------
# Candidate Regression Algorithms
# ---------------------------------------------------------

# These are the three algorithms that will be compared.
CANDIDATES = {

    # Gradient Boosting regression model
    "GradientBoostingRegressor": GradientBoostingRegressor,

    # Random Forest regression model
    "RandomForestRegressor": RandomForestRegressor,

    # XGBoost regression model
    "XGBRegressor": XGBRegressor,
}


# ---------------------------------------------------------
# Hyperparameter Grid
# ---------------------------------------------------------

# Different combinations of hyperparameters
# will be tested for each algorithm.
GRID = {

    # Hyperparameters for Gradient Boosting
    "GradientBoostingRegressor": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
    },

    # Hyperparameters for Random Forest
    "RandomForestRegressor": {
        "n_estimators": [100, 250],
        "max_depth": [None, 10],
        "min_samples_leaf": [1, 4],
    },

    # Hyperparameters for XGBoost
    "XGBRegressor": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
        "subsample": [0.8, 1.0],
    },
}


# ---------------------------------------------------------
# Build Training Dataset
# ---------------------------------------------------------

def build_training_frame() -> pd.DataFrame:

    # Load monthly user financial data
    monthly = load_user_monthly()

    # Create monthly features from the raw data
    feats = build_monthly_features(monthly)

    # Add future savings target
    # This is the value that the model needs to predict.
    feats = add_future_targets(feats)

    # Remove rows where any required feature
    # or future_savings target is missing.
    return feats.dropna(
        subset=FEATURES + ["future_savings"]
    )


# ---------------------------------------------------------
# Main Training Pipeline
# ---------------------------------------------------------

def main() -> None:

    # Build the complete training dataset
    df = build_training_frame()


    # -----------------------------------------------------
    # Split data into Training, Validation and Test sets
    # -----------------------------------------------------

    # Training:
    # Used to train the models.
    #
    # Validation:
    # Used to compare models and select hyperparameters.
    #
    # Test:
    # Used only for final evaluation.
    train, val, test = split_by_ym(df)


    # -----------------------------------------------------
    # Prepare Training Data
    # -----------------------------------------------------

    # Xtr = training features
    # ytr = training target
    #
    # signed_log1p() transforms the savings values
    # before training.
    Xtr = train[FEATURES]

    ytr = signed_log1p(
        train["future_savings"]
    )


    # -----------------------------------------------------
    # Prepare Validation Data
    # -----------------------------------------------------

    Xva = val[FEATURES]

    yva = signed_log1p(
        val["future_savings"]
    )


    # -----------------------------------------------------
    # Prepare Test Data
    # -----------------------------------------------------

    Xte = test[FEATURES]

    # Keep the original test values because we want
    # to calculate final metrics in the original scale.
    yte_orig = test[
        "future_savings"
    ].astype(float)


    # -----------------------------------------------------
    # Compare Candidate Algorithms
    # -----------------------------------------------------

    # Dictionary to store results of each algorithm
    results = {}


    # Loop through every candidate algorithm
    for name, est in CANDIDATES.items():

        # Search through all hyperparameter combinations
        # and select the combination with the lowest
        # validation MAE.
        model, params, mae = search_params(

            # Current algorithm
            est,

            # Hyperparameters for this algorithm
            GRID[name],

            # Training features and target
            Xtr,
            ytr,

            # Validation features and target
            Xva,
            yva,

            # Evaluation metric
            #
            # MAE = Mean Absolute Error
            #
            # Lower MAE is better.
            metric_fn=lambda yt, yp:
                regression_metrics(
                    yt,
                    yp
                )["MAE"],

            # We want to minimize MAE
            maximize=False
        )


        # Store the results for this algorithm
        results[name] = {
            "val_MAE_log": mae,
            "params": params
        }


    # -----------------------------------------------------
    # Select the Best Algorithm
    # -----------------------------------------------------

    # Select the algorithm with the lowest validation MAE.
    best_name = min(
        results,
        key=lambda k:
            results[k]["val_MAE_log"]
    )


    # Get the best hyperparameters
    # of the selected algorithm.
    params = results[
        best_name
    ]["params"]


    # -----------------------------------------------------
    # Create Final Model
    # -----------------------------------------------------

    # Create a new model using:
    #
    # 1. Best algorithm
    # 2. Best hyperparameters
    model = CANDIDATES[
        best_name
    ](**params)


    # -----------------------------------------------------
    # Final Model Training
    # -----------------------------------------------------

    # Combine training + validation data.
    #
    # Validation data was already used for selecting
    # the best algorithm and hyperparameters.
    #
    # Therefore, we can now use both datasets
    # to train the final model.
    model.fit(
        pd.concat(
            [Xtr, Xva],
            axis=0
        ),
        np.concatenate(
            [ytr, yva]
        )
    )


    # -----------------------------------------------------
    # Test Set Evaluation
    # -----------------------------------------------------

    # Make predictions on completely unseen test data.
    test_predictions = signed_expm1(
        model.predict(Xte)
    )


    # Calculate final test metrics.
    #
    # Metrics include:
    # MAE
    # RMSE
    # R2
    test_metrics = regression_metrics(
        yte_orig,
        test_predictions
    )


    # -----------------------------------------------------
    # Validation Set Evaluation
    # -----------------------------------------------------

    # Make predictions on validation data.
    val_predictions = signed_expm1(
        model.predict(Xva)
    )


    # Calculate validation metrics.
    val_metrics = regression_metrics(
        signed_expm1(yva),
        val_predictions
    )


    # -----------------------------------------------------
    # Create Algorithm Comparison
    # -----------------------------------------------------

    # Store validation MAE and selected parameters
    # for every algorithm.
    comparison = {
        k: {
            "val_MAE_log": v["val_MAE_log"],
            "final_params": v["params"]
        }

        for k, v in results.items()
    }


    # -----------------------------------------------------
    # Save Model and Metadata
    # -----------------------------------------------------



# ---------------------------------------------------------
# Deep Learning Models (Regression)
# ---------------------------------------------------------

# Scale features for DL (StandardScaler fitted on TRAIN only)
scaler_dl = StandardScaler().fit(Xtr)
Xtr_dl = scaler_dl.transform(Xtr)
Xva_dl = scaler_dl.transform(Xva)
Xte_dl = scaler_dl.transform(Xte)

# Prepare y arrays for DL
ytr = signed_log1p(train["future_savings"])
yva = signed_log1p(val["future_savings"])
yte_orig = test["future_savings"].astype(float)

# Convert to numpy
ytr_np = ytr.values if hasattr(ytr, 'values') else np.array(ytr)
yva_np = yva.values if hasattr(yva, 'values') else np.array(yva)

input_dim = Xtr_dl.shape[1]

print(f"\n=== Training Deep Learning Models for Savings Prediction ===")
print(f"Input dim: {input_dim} | Train: {len(Xtr_dl)} | Val: {len(Xva_dl)} | Test: {len(Xte_dl)}")

# ---- MLP ----
print("\n--- MLP ---")
mlp_reg = MLPRegressor(input_dim, hidden_dims=(128, 64, 32), dropout=0.2)
mlp_reg, mlp_history = train_dl_model(
    mlp_reg, Xtr_dl, ytr_np, Xva_dl, yva_np,
    task='regression', epochs=100, batch_size=256, lr=1e-3, patience=15
)
mlp_metrics, mlp_preds = evaluate_dl_regression(mlp_reg, Xte_dl, yte_orig)
print(f"MLP Test: MAE={mlp_metrics['MAE']:.2f}, RMSE={mlp_metrics['RMSE']:.2f}, R2={mlp_metrics['R2']:.4f}")

# ---- ResNet-style MLP ----
print("\n--- ResNet MLP ---")
resnet_reg = ResNetMLPRegressor(input_dim, hidden_dim=128, n_blocks=3, dropout=0.2)
resnet_reg, resnet_history = train_dl_model(
    resnet_reg, Xtr_dl, ytr_np, Xva_dl, yva_np,
    task='regression', epochs=100, batch_size=256, lr=1e-3, patience=15
)
resnet_metrics, resnet_preds = evaluate_dl_regression(resnet_reg, Xte_dl, yte_orig)
print(f"ResNet MLP Test: MAE={resnet_metrics['MAE']:.2f}, RMSE={resnet_metrics['RMSE']:.2f}, R2={resnet_metrics['R2']:.4f}")

# ---------------------------------------------------------
# Combine DL results with traditional ML comparison
# ---------------------------------------------------------

# Add DL models to comparison
comparison["MLP"] = {"test_MAE": mlp_metrics['MAE'], "test_RMSE": mlp_metrics['RMSE'], "test_R2": mlp_metrics['R2'], "params": "hidden=(128,64,32), dropout=0.2"}
comparison["ResNet_MLP"] = {"test_MAE": resnet_metrics['MAE'], "test_RMSE": resnet_metrics['RMSE'], "test_R2": resnet_metrics['R2'], "params": "hidden=128, blocks=3, dropout=0.2"}

# Save DL model artifacts
import joblib
joblib.dump(mlp_reg.state_dict(), ARTIFACTS_DIR / "savings" / "mlp_state_dict.pt")
joblib.dump(scaler_dl, ARTIFACTS_DIR / "savings" / "scaler_dl.pkl")
joblib.dump(resnet_reg.state_dict(), ARTIFACTS_DIR / "savings" / "resnet_state_dict.pt")

# Update metrics dict for saving
metrics["test_dl"] = {
    "MLP": mlp_metrics,
    "ResNet_MLP": resnet_metrics
}
metrics["algorithm_comparison"] = comparison


    save_predict_api_meta(

        # Name of the model
        name=RUN_NAME,

        # Final trained model
        model=model,

        # Features used by the model
        features=FEATURES,

        # Folder where artifacts will be saved
        art_dir=ARTIFACTS_DIR / RUN_NAME,

        # Save validation metrics,
        # test metrics and algorithm comparison
        metrics={
            "val": val_metrics,
            "test": test_metrics,
            "algorithm_comparison": comparison
        },

        # Name of the selected algorithm
        algorithm=best_name,

        # Model version
        version="1.0",

        # Target variable
        target="future_savings",

        # Description of what the model predicts
        target_desc=(
            "Predicted future-month aggregate savings "
            "(aggregate income - aggregate expense)."
        ),
    )


# ---------------------------------------------------------
# Run the training pipeline
# ---------------------------------------------------------

# In Google Colab, we can directly call main().
main()

/tmp/ipykernel_7587/3169358447.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(mat, axis=1)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_7587/3169358447.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(mat, axis=1)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Artifacts saved under -> /content/artifacts/savings


**Train Wellness model**

In [15]:
from __future__ import annotations

# ---------------------------------------------------------
# Import required libraries
# ---------------------------------------------------------

import pandas as pd

# Import regression algorithms
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)

from xgboost import XGBRegressor


# ---------------------------------------------------------
# Model / Training Run Configuration
# ---------------------------------------------------------

# Name of this ML training run
RUN_NAME = "wellness"


# ---------------------------------------------------------
# Features used to calculate/predict wellness score
# ---------------------------------------------------------

FEATURES = [
    # Basic financial features
    "income",
    "expense",
    "cash_flow",

    # Financial ratio features
    "expense_ratio",
    "savings_ratio",
    "rent_burden",

    # Stability features
    "spending_stability",
    "income_stability",

    # 3-month average features
    "savings_3m_mean",
    "cash_flow_3m_mean",
    "income_3m_mean",
    "expense_3m_mean",

    # Trend features
    "income_trend",
    "expense_trend",

    # Transaction features
    "txn_count",
    "avg_txn_amount",

    # Time features
    "month",
    "year",
]


# ---------------------------------------------------------
# Candidate Regression Algorithms
# ---------------------------------------------------------

# These algorithms will be compared to find the
# best model for predicting the wellness score.
CANDIDATES = {

    # XGBoost regression model
    "XGBRegressor": XGBRegressor,

    # Random Forest regression model
    "RandomForestRegressor": RandomForestRegressor,

    # Gradient Boosting regression model
    "GradientBoostingRegressor": GradientBoostingRegressor,
}


# ---------------------------------------------------------
# Hyperparameter Grid
# ---------------------------------------------------------

# Different hyperparameter combinations will be tested
# for each candidate algorithm.
GRID = {

    # Hyperparameters for XGBoost
    "XGBRegressor": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
        "subsample": [0.8, 1.0],
    },

    # Hyperparameters for Random Forest
    "RandomForestRegressor": {
        "n_estimators": [100, 250],
        "max_depth": [None, 10],
        "min_samples_leaf": [1, 4],
    },

    # Hyperparameters for Gradient Boosting
    "GradientBoostingRegressor": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
    },
}


# ---------------------------------------------------------
# Build Training Dataset
# ---------------------------------------------------------

def build_training_frame() -> pd.DataFrame:

    # Load monthly user financial data
    annual = load_user_monthly()

    # Create financial features from the monthly data
    feats = build_monthly_features(annual)

    # Calculate the wellness score using predefined rules
    # based on the financial features.
    feats["wellness_score"] = wellness_score(feats)

    # Remove rows where any required feature is missing
    return feats.dropna(
        subset=FEATURES
    )


# ---------------------------------------------------------
# Main Training Pipeline
# ---------------------------------------------------------

def main() -> None:

    # Build the complete dataset
    df = build_training_frame()


    # -----------------------------------------------------
    # Split data into Training, Validation and Test sets
    # -----------------------------------------------------

    # Training data:
    # Used to train the candidate models.
    #
    # Validation data:
    # Used to compare algorithms and hyperparameters.
    #
    # Test data:
    # Used for final evaluation.
    train, val, test = split_by_ym(df)


    # -----------------------------------------------------
    # Prepare Training Data
    # -----------------------------------------------------

    # Select training features
    Xtr = train[FEATURES]

    # Select training target
    # Convert wellness score to integer
    ytr = train[
        "wellness_score"
    ].astype(int)


    # -----------------------------------------------------
    # Prepare Validation Data
    # -----------------------------------------------------

    Xva = val[FEATURES]

    yva = val[
        "wellness_score"
    ].astype(int)


    # -----------------------------------------------------
    # Prepare Test Data
    # -----------------------------------------------------

    Xte = test[FEATURES]

    yte = test[
        "wellness_score"
    ].astype(int)


    # -----------------------------------------------------
    # Compare Candidate Algorithms
    # -----------------------------------------------------

    # Dictionary to store the performance of every model
    results = {}


    # Loop through all candidate algorithms
    for name, est in CANDIDATES.items():

        # Search through different hyperparameter
        # combinations for the current algorithm.
        model, params, mae = search_params(

            # Current algorithm
            est,

            # Hyperparameter grid for current algorithm
            GRID[name],

            # Training features and target
            Xtr,
            ytr,

            # Validation features and target
            Xva,
            yva,

            # Use MAE as the evaluation metric
            #
            # Lower MAE means better predictions.
            metric_fn=lambda yt, yp:
                regression_metrics(
                    yt,
                    yp
                )["MAE"],

            # We want to minimize MAE
            maximize=False
        )


        # Store the validation MAE
        # and best parameters for this algorithm.
        results[name] = {
            "val_MAE": mae,
            "params": params
        }


    # -----------------------------------------------------
    # Select the Best Algorithm
    # -----------------------------------------------------

    # Select the model with the lowest validation MAE
    best_name = min(
        results,
        key=lambda k:
            results[k]["val_MAE"]
    )


    # Get the best hyperparameters
    # for the selected algorithm.
    params = results[
        best_name
    ]["params"]


    # -----------------------------------------------------
    # Create Final Model
    # -----------------------------------------------------

    # Create a new model using the best algorithm
    # and its best hyperparameters.
    model = CANDIDATES[
        best_name
    ](**params)


    # -----------------------------------------------------
    # Final Model Training
    # -----------------------------------------------------

    # Combine training and validation data.
    #
    # The validation set was already used to select
    # the best algorithm and hyperparameters.
    #
    # Therefore, we can now use both training and
    # validation data to train the final model.
    model.fit(
        pd.concat(
            [Xtr, Xva]
        ),
        pd.concat(
            [ytr, yva]
        )
    )


    # -----------------------------------------------------
    # Test Set Evaluation
    # -----------------------------------------------------

    # Predict wellness scores for unseen test data
    test_predictions = model.predict(Xte)

    # Calculate test performance
    test_metrics = regression_metrics(
        yte,
        test_predictions
    )


    # -----------------------------------------------------
    # Validation Set Evaluation
    # -----------------------------------------------------

    # Predict wellness scores for validation data
    val_predictions = model.predict(Xva)

    # Calculate validation performance
    val_metrics = regression_metrics(
        yva,
        val_predictions
    )


    # -----------------------------------------------------
    # Create Algorithm Comparison
    # -----------------------------------------------------

    # Store validation MAE and final parameters
    # for every algorithm that was tested.
    comparison = {
        k: {
            "val_MAE": v["val_MAE"],
            "final_params": v["params"]
        }

        for k, v in results.items()
    }


    # -----------------------------------------------------
    # Save Model and Metadata
    # -----------------------------------------------------



# ---------------------------------------------------------
# Deep Learning Models (Regression)
# ---------------------------------------------------------

# Scale features for DL (StandardScaler fitted on TRAIN only)
scaler_dl = StandardScaler().fit(Xtr)
Xtr_dl = scaler_dl.transform(Xtr)
Xva_dl = scaler_dl.transform(Xva)
Xte_dl = scaler_dl.transform(Xte)

# Prepare y arrays for DL
ytr = train["wellness_score"].astype(float)
yva = val["wellness_score"].astype(float)
yte_orig = test["wellness_score"].astype(float)

# Convert to numpy
ytr_np = ytr.values if hasattr(ytr, 'values') else np.array(ytr)
yva_np = yva.values if hasattr(yva, 'values') else np.array(yva)

input_dim = Xtr_dl.shape[1]

print(f"\n=== Training Deep Learning Models for Wellness Prediction ===")
print(f"Input dim: {input_dim} | Train: {len(Xtr_dl)} | Val: {len(Xva_dl)} | Test: {len(Xte_dl)}")

# ---- MLP ----
print("\n--- MLP ---")
mlp_reg = MLPRegressor(input_dim, hidden_dims=(128, 64, 32), dropout=0.2)
mlp_reg, mlp_history = train_dl_model(
    mlp_reg, Xtr_dl, ytr_np, Xva_dl, yva_np,
    task='regression', epochs=100, batch_size=256, lr=1e-3, patience=15
)
mlp_metrics, mlp_preds = evaluate_dl_regression(mlp_reg, Xte_dl, yte_orig)
print(f"MLP Test: MAE={mlp_metrics['MAE']:.2f}, RMSE={mlp_metrics['RMSE']:.2f}, R2={mlp_metrics['R2']:.4f}")

# ---- ResNet-style MLP ----
print("\n--- ResNet MLP ---")
resnet_reg = ResNetMLPRegressor(input_dim, hidden_dim=128, n_blocks=3, dropout=0.2)
resnet_reg, resnet_history = train_dl_model(
    resnet_reg, Xtr_dl, ytr_np, Xva_dl, yva_np,
    task='regression', epochs=100, batch_size=256, lr=1e-3, patience=15
)
resnet_metrics, resnet_preds = evaluate_dl_regression(resnet_reg, Xte_dl, yte_orig)
print(f"ResNet MLP Test: MAE={resnet_metrics['MAE']:.2f}, RMSE={resnet_metrics['RMSE']:.2f}, R2={resnet_metrics['R2']:.4f}")

# ---------------------------------------------------------
# Combine DL results with traditional ML comparison
# ---------------------------------------------------------

# Add DL models to comparison
comparison["MLP"] = {"test_MAE": mlp_metrics['MAE'], "test_RMSE": mlp_metrics['RMSE'], "test_R2": mlp_metrics['R2'], "params": "hidden=(128,64,32), dropout=0.2"}
comparison["ResNet_MLP"] = {"test_MAE": resnet_metrics['MAE'], "test_RMSE": resnet_metrics['RMSE'], "test_R2": resnet_metrics['R2'], "params": "hidden=128, blocks=3, dropout=0.2"}

# Save DL model artifacts
import joblib
joblib.dump(mlp_reg.state_dict(), ARTIFACTS_DIR / "wellness" / "mlp_state_dict.pt")
joblib.dump(scaler_dl, ARTIFACTS_DIR / "wellness" / "scaler_dl.pkl")
joblib.dump(resnet_reg.state_dict(), ARTIFACTS_DIR / "wellness" / "resnet_state_dict.pt")

# Update metrics dict for saving
metrics["test_dl"] = {
    "MLP": mlp_metrics,
    "ResNet_MLP": resnet_metrics
}
metrics["algorithm_comparison"] = comparison


    save_predict_api_meta(

        # Name of this model
        name=RUN_NAME,

        # Final trained model
        model=model,

        # Features used by the model
        features=FEATURES,

        # Directory where model artifacts are stored
        art_dir=ARTIFACTS_DIR / RUN_NAME,

        # Save validation metrics,
        # test metrics and algorithm comparison
        metrics={
            "val": val_metrics,
            "test": test_metrics,
            "algorithm_comparison": comparison
        },

        # Name of selected algorithm
        algorithm=best_name,

        # Model version
        version="1.0",

        # Target variable
        target="wellness_score",

        # Description of the target
        target_desc=(
            "Rule-derived evaluation score representing "
            "consumer financial wellness."
        ),
    )


# ---------------------------------------------------------
# Run the training pipeline
# ---------------------------------------------------------

# In Google Colab, we can directly call main().
main()

/tmp/ipykernel_7587/3169358447.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(mat, axis=1)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_7587/3169358447.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(mat, axis=1)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Artifacts saved under -> /content/artifacts/wellness


**Train risk model**

In [16]:
from __future__ import annotations

# ---------------------------------------------------------
# Import required libraries
# ---------------------------------------------------------

import numpy as np
import pandas as pd

# Import classification algorithms
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Import XGBoost classification algorithm
from xgboost import XGBClassifier


# ---------------------------------------------------------
# Model / Training Run Configuration
# ---------------------------------------------------------

# Name of this ML training run
RUN_NAME = "risk"


# ---------------------------------------------------------
# Features used for Risk Prediction
# ---------------------------------------------------------

FEATURES = [
    # Basic financial features
    "income",
    "expense",
    "cash_flow",
    "expense_ratio",

    # Savings features
    "savings",
    "savings_ratio",

    # Historical cash-flow features
    "cash_flow_1m",
    "cash_flow_3m_mean",

    # Historical income features
    "income_1m",
    "income_2m",
    "income_3m_mean",

    # Historical expense features
    "expense_1m",
    "expense_2m",
    "expense_3m_mean",

    # Trend features
    "expense_trend",
    "income_trend",

    # Stability features
    "spending_stability",
    "income_stability",

    # Transaction features
    "txn_count",
    "avg_txn_amount",

    # Time features
    "month",
    "year",
]


# ---------------------------------------------------------
# Candidate Classification Algorithms
# ---------------------------------------------------------

# These three classification algorithms will be compared
# to find the best model for financial risk prediction.
CANDIDATES = {

    # Logistic Regression
    "LogisticRegression": LogisticRegression,

    # Random Forest Classifier
    "RandomForestClassifier": RandomForestClassifier,

    # XGBoost Classifier
    "XGBClassifier": XGBClassifier,
}


# ---------------------------------------------------------
# Hyperparameter Grid
# ---------------------------------------------------------

# Different hyperparameter combinations that will be tested
# for each candidate algorithm.
GRID = {

    # Hyperparameters for Logistic Regression
    "LogisticRegression": {
        "C": [0.1, 1.0, 10.0],
        "class_weight": [None, "balanced"],
        "max_iter": [5000],
    },

    # Hyperparameters for Random Forest
    "RandomForestClassifier": {
        "n_estimators": [100, 250],
        "max_depth": [None, 10],
        "min_samples_leaf": [1, 4],
    },

    # Hyperparameters for XGBoost
    "XGBClassifier": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
        "subsample": [0.8, 1.0],
    },
}


# ---------------------------------------------------------
# Build Training Dataset
# ---------------------------------------------------------

def build_training_frame() -> pd.DataFrame:

    # Load monthly user financial data
    monthly = load_user_monthly()

    # Create financial features from the monthly data
    feats = build_monthly_features(monthly)

    # Create the risk target:
    # 0 = Low Risk
    # 1 = High Risk
    feats["risk"] = risk_flag(feats)


    # -----------------------------------------------------
    # Keep only records starting from month 6
    # -----------------------------------------------------

    # cumcount() counts records for each user starting from 0.
    #
    # >= 5 means:
    # 0,1,2,3,4 -> removed
    # 5,6,7,...  -> retained
    #
    # This ensures that enough historical data is available
    # for stable risk calculations.
    retained = (
        feats.groupby(
            "user_id",
            sort=False
        ).cumcount() >= 5
    )


    # Keep only retained records and remove rows
    # where required features are missing.
    return feats[
        retained
    ].dropna(
        subset=FEATURES
    )


# ---------------------------------------------------------
# Calculate ROC-AUC
# ---------------------------------------------------------

def _roc_auc(
    y_true,
    y_pred_prob
):

    # Import ROC-AUC metric
    from sklearn.metrics import roc_auc_score

    # Calculate ROC-AUC using predicted probabilities
    return roc_auc_score(
        y_true,
        y_pred_prob
    )


# ---------------------------------------------------------
# Main Training Pipeline
# ---------------------------------------------------------

def main() -> None:

    # Build the complete training dataset
    df = build_training_frame()


    # -----------------------------------------------------
    # Split Data
    # -----------------------------------------------------

    # Training:
    # Used to train candidate models.
    #
    # Validation:
    # Used to compare algorithms and hyperparameters.
    #
    # Test:
    # Used for final evaluation on unseen data.
    train, val, test = split_by_ym(df)


    # -----------------------------------------------------
    # Prepare Training Data
    # -----------------------------------------------------

    # Training features
    Xtr = train[FEATURES]

    # Training target
    # 0 = Low Risk
    # 1 = High Risk
    ytr = train[
        "risk"
    ].astype(int)


    # -----------------------------------------------------
    # Prepare Validation Data
    # -----------------------------------------------------

    Xva = val[FEATURES]

    yva = val[
        "risk"
    ].astype(int)


    # -----------------------------------------------------
    # Prepare Test Data
    # -----------------------------------------------------

    Xte = test[FEATURES]

    yte = test[
        "risk"
    ].astype(int)


    # -----------------------------------------------------
    # Feature Scaling
    # -----------------------------------------------------

    # StandardScaler standardizes features so that they
    # have approximately:
    #
    # Mean = 0
    # Standard Deviation = 1
    #
    # IMPORTANT:
    # The scaler is fitted ONLY on training data.
    # This prevents data leakage.
    scaler = StandardScaler().fit(Xtr)


    # Transform training, validation and test features
    # using the same scaler.
    Xtr_s = scaler.transform(Xtr)
    Xva_s = scaler.transform(Xva)
    Xte_s = scaler.transform(Xte)


    # -----------------------------------------------------
    # Compare Candidate Algorithms
    # -----------------------------------------------------

    # Store results of every algorithm
    results = {}


    # -----------------------------------------------------
    # Probability Prediction Function
    # -----------------------------------------------------

    # ROC-AUC requires probability scores rather than
    # only the final predicted class.
    #
    # predict_proba()[:, 1] gives the probability
    # of the positive class:
    #
    # 0 = Low Risk
    # 1 = High Risk
    proba_predict = lambda m, X: (
        m.predict_proba(X)[:, 1]
    )


    # -----------------------------------------------------
    # Train and Evaluate Every Candidate Model
    # -----------------------------------------------------

    for name, est in CANDIDATES.items():

        # Search through all hyperparameter combinations
        # and select the combination with the highest
        # validation ROC-AUC.
        model, params, auc = search_params(

            # Current algorithm
            est,

            # Hyperparameter grid for current algorithm
            GRID[name],

            # Training data
            Xtr_s,
            ytr,

            # Validation data
            Xva_s,
            yva,

            # ROC-AUC evaluation metric
            metric_fn=lambda yt, yp:
                _roc_auc(
                    yt,
                    yp
                ),

            # Higher ROC-AUC is better
            maximize=True,

            # Use predicted probabilities for ROC-AUC
            predict_fn=proba_predict,
        )


        # Store validation ROC-AUC and best parameters
        results[name] = {
            "val_ROC_AUC": auc,
            "params": params
        }


    # -----------------------------------------------------
    # Select Best Algorithm
    # -----------------------------------------------------

    # Select the algorithm with the highest validation
    # ROC-AUC score.
    best_name = max(
        results,
        key=lambda k:
            results[k]["val_ROC_AUC"]
    )


    # Get the best hyperparameters
    # of the selected algorithm.
    params = results[
        best_name
    ]["params"]


    # -----------------------------------------------------
    # Create Final Model
    # -----------------------------------------------------

    # Create a new model using:
    #
    # 1. Best algorithm
    # 2. Best hyperparameters
    model = CANDIDATES[
        best_name
    ](**params)


    # -----------------------------------------------------
    # Final Model Training
    # -----------------------------------------------------

    # Combine training and validation data.
    #
    # Validation data has already been used to select
    # the best model and hyperparameters.
    #
    # Therefore, it can now be included in final training.
    model.fit(
        np.vstack(
            [Xtr_s, Xva_s]
        ),
        np.concatenate(
            [ytr, yva]
        )
    )


    # -----------------------------------------------------
    # Test Predictions
    # -----------------------------------------------------

    # Predict final risk class on unseen test data.
    pred = model.predict(
        Xte_s
    )


    # Generate probability of High Risk (class 1)
    # if the model supports predict_proba().
    proba = (
        model.predict_proba(Xte_s)[:, 1]
        if hasattr(
            model,
            "predict_proba"
        )
        else None
    )


    # -----------------------------------------------------
    # Test Evaluation
    # -----------------------------------------------------

    # Calculate:
    #
    # Accuracy
    # Precision
    # Recall
    # F1 Score
    # ROC-AUC
    # Confusion Matrix
    test_metrics = classification_metrics(
        yte,
        pred,
        proba
    )


    # -----------------------------------------------------
    # Validation Evaluation
    # -----------------------------------------------------

    # Generate validation predictions
    val_pred = model.predict(
        Xva_s
    )


    # Generate validation probabilities
    #
    # If test probabilities were available, then the
    # selected model supports predict_proba().
    val_proba = (
        model.predict_proba(Xva_s)[:, 1]
        if proba is not None
        else None
    )


    # Calculate validation metrics
    val_metrics = classification_metrics(
        yva,
        val_pred,
        val_proba
    )


    # -----------------------------------------------------
    # Create Algorithm Comparison
    # -----------------------------------------------------

    # Store validation ROC-AUC and final parameters
    # for every algorithm that was tested.
    comparison = {
        k: {
            "val_ROC_AUC": v["val_ROC_AUC"],
            "final_params": v["params"]
        }

        for k, v in results.items()
    }


    # -----------------------------------------------------
    # Save Model and Metadata
    # -----------------------------------------------------



# ---------------------------------------------------------
# Deep Learning Models (Classification)
# ---------------------------------------------------------

# Scale features for DL (StandardScaler fitted on TRAIN only)
scaler_dl = StandardScaler().fit(Xtr)
Xtr_dl = scaler_dl.transform(Xtr)
Xva_dl = scaler_dl.transform(Xva)
Xte_dl = scaler_dl.transform(Xte)

# Prepare y arrays for DL
ytr = train["risk"].astype(int)
yva = val["risk"].astype(int)
yte = test["risk"].astype(int)

ytr_np = ytr.values
yva_np = yva.values
yte_np = yte.values

input_dim = Xtr_dl.shape[1]

print(f"\n=== Training Deep Learning Models for Risk Prediction ===")
print(f"Input dim: {input_dim} | Train: {len(Xtr_dl)} | Val: {len(Xva_dl)} | Test: {len(Xte_dl)}")

# ---- MLP Classifier ----
print("\n--- MLP Classifier ---")
mlp_clf = MLPClassifier(input_dim, hidden_dims=(128, 64, 32), dropout=0.2)
mlp_clf, mlp_history = train_dl_model(
    mlp_clf, Xtr_dl, ytr_np, Xva_dl, yva_np,
    task='classification', epochs=100, batch_size=256, lr=1e-3, patience=15
)
mlp_metrics, mlp_preds, mlp_probs = evaluate_dl_classification(mlp_clf, Xte_dl, yte_np)
print(f"MLP Test: Acc={mlp_metrics['Accuracy']:.4f}, F1={mlp_metrics['F1_weighted']:.4f}, ROC-AUC={mlp_metrics['ROC_AUC']}")

# ---- ResNet-style MLP Classifier ----
print("\n--- ResNet MLP Classifier ---")
resnet_clf = ResNetMLPClassifier(input_dim, hidden_dim=128, n_blocks=3, dropout=0.2)
resnet_clf, resnet_history = train_dl_model(
    resnet_clf, Xtr_dl, ytr_np, Xva_dl, yva_np,
    task='classification', epochs=100, batch_size=256, lr=1e-3, patience=15
)
resnet_metrics, resnet_preds, resnet_probs = evaluate_dl_classification(resnet_clf, Xte_dl, yte_np)
print(f"ResNet MLP Test: Acc={resnet_metrics['Accuracy']:.4f}, F1={resnet_metrics['F1_weighted']:.4f}, ROC-AUC={resnet_metrics['ROC_AUC']}")

# ---------------------------------------------------------
# Combine DL results with traditional ML comparison
# ---------------------------------------------------------

comparison["MLP"] = {"test_Accuracy": mlp_metrics['Accuracy'], "test_F1": mlp_metrics['F1_weighted'], "test_ROC_AUC": mlp_metrics['ROC_AUC'], "params": "hidden=(128,64,32), dropout=0.2"}
comparison["ResNet_MLP"] = {"test_Accuracy": resnet_metrics['Accuracy'], "test_F1": resnet_metrics['F1_weighted'], "test_ROC_AUC": resnet_metrics['ROC_AUC'], "params": "hidden=128, blocks=3, dropout=0.2"}

# Save DL model artifacts
import joblib
joblib.dump(mlp_clf.state_dict(), ARTIFACTS_DIR / "risk" / "mlp_state_dict.pt")
joblib.dump(scaler_dl, ARTIFACTS_DIR / "risk" / "scaler_dl.pkl")
joblib.dump(resnet_clf.state_dict(), ARTIFACTS_DIR / "risk" / "resnet_state_dict.pt")

# Update metrics dict for saving
metrics["test_dl"] = {
    "MLP": mlp_metrics,
    "ResNet_MLP": resnet_metrics
}
metrics["algorithm_comparison"] = comparison


    save_predict_api_meta(

        # Model name
        name=RUN_NAME,

        # Final trained model
        model=model,

        # Features used by the model
        features=FEATURES,

        # Directory where artifacts are saved
        art_dir=ARTIFACTS_DIR / RUN_NAME,

        # Save validation metrics,
        # test metrics and algorithm comparison
        metrics={
            "val": val_metrics,
            "test": test_metrics,
            "algorithm_comparison": comparison
        },

        # Name of selected algorithm
        algorithm=best_name,

        # Model version
        version="1.0",

        # Target variable
        target="risk",

        # Description of target
        target_desc=(
            "Binary financial risk warning flag "
            "(0=Low, 1=High Risk)."
        ),

        # Mapping between numeric labels and
        # human-readable risk categories
        label_mapping={
            "0": "Low Risk",
            "1": "High Risk"
        },

        # Save the fitted scaler along with the model
        scaler=scaler,
    )


# ---------------------------------------------------------
# Run the training pipeline
# ---------------------------------------------------------

# In Google Colab, we can directly call main().
main()

/tmp/ipykernel_7587/3169358447.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(mat, axis=1)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_7587/3169358447.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(mat, axis=1)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Artifacts saved under -> /content/artifacts/risk


In [17]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

RUN_NAME = "payment"
LABELS = ["UPI", "Debit Card", "Cash", "Bank Transfer"]

CANDIDATES = {
    "XGBClassifier": XGBClassifier,
    "RandomForestClassifier": RandomForestClassifier,
    "LogisticRegression": LogisticRegression,
}

GRID = {
    "XGBClassifier": {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 4], "subsample": [0.8, 1.0]},
    "RandomForestClassifier": {"n_estimators": [100, 250], "max_depth": [None, 10], "min_samples_leaf": [1, 4]},
    "LogisticRegression": {"C": [0.1, 1.0, 10.0], "max_iter": [2000], "solver": ["lbfgs"]},
}

def _build() -> pd.DataFrame:
    txn = load_transactions()
    f = build_transaction_features(txn)

    # Target encoding alignment
    cat_dummies = pd.get_dummies(f["category"], prefix="txn_cat", dtype=int)
    for c in sorted(cat_dummies.columns):
        f[c] = 0
    for c in cat_dummies.columns:
        f.loc[cat_dummies.index, c] = cat_dummies.loc[cat_dummies.index, c]

    base_cols = [
        "amount", "month", "day_of_month", "is_weekend",
        "month_cash_flow_so_far", "month_income_so_far",
        "hist_UPI", "hist_Debit_Card", "hist_Cash", "hist_Bank_Transfer", "hist_total",
    ]
    cum_cat = sorted([c for c in f.columns if c.startswith("cat_cum_")])
    feat_cols = base_cols + sorted(cat_dummies.columns) + cum_cat
    f["date"] = pd.to_datetime(f["date"])
    return f[feat_cols + ["date", "payment_mode"]]

def main() -> None:
    f = _build()
    feats = [c for c in f.columns if c not in ("date", "payment_mode")]

    train, val, test = split_transactions_by_date(f)

    def _xy(sub):
        return sub[feats], sub["payment_mode"].astype(str)

    Xtr, ytr = _xy(train)
    Xva, yva = _xy(val)
    Xte, yte = _xy(test)

    # Local categorical encoder fitting strictly to training sets
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder().fit(ytr)
    ytr_e, yva_e, yte_e = le.transform(ytr), le.transform(yva), le.transform(yte)
    label_mapping = {str(i): cls for i, cls in enumerate(le.classes_)}

    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xva_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xva), scaler.transform(Xte)

    results = {}
    for name, est in CANDIDATES.items():
        model, params, f1 = search_params(
            est, GRID[name], Xtr_s, ytr_e, Xva_s, yva_e,
            metric_fn=lambda yt, yp: classification_metrics(yt, yp)["F1_weighted"], maximize=True,
        )
        results[name] = {"val_F1": f1, "params": params}

    best_name = max(results, key=lambda k: results[k]["val_F1"])
    params = results[best_name]["params"]

    model = CANDIDATES[best_name](**params)
    model.fit(np.vstack([Xtr_s, Xva_s]), np.concatenate([ytr_e, yva_e]))

    pred_e = model.predict(Xte_s)
    test_metrics = classification_metrics(yte_e, pred_e, None)
    if "Confusion_Matrix" in test_metrics:
        del test_metrics["Confusion_Matrix"]

    from sklearn.metrics import classification_report
    report = classification_report(yte_e, pred_e, output_dict=True, zero_division=0, target_names=le.classes_)
    per_class = {k: {kk: round(vv, 4) for kk, vv in v.items() if isinstance(v, dict) and kk != "support"}
                 for k, v in report.items() if k in le.classes_}

    comparison = {k: {"val_F1": v["val_F1"], "final_params": v["params"]} for k, v in results.items()}
    save_predict_api_meta(
        name=RUN_NAME,
        model=model,
        features=feats,
        art_dir=ARTIFACTS_DIR / RUN_NAME,
        metrics={
            "val": {"F1_weighted": results[best_name]["val_F1"]},
            "test": test_metrics,
            "test_per_class": per_class,
            "algorithm_comparison": comparison
        },
        algorithm=best_name,
        version="1.0",
        target="payment_mode",
        target_desc="Best payment mode behavior label based on user characteristics.",
        scaler=scaler,
        label_mapping=label_mapping,
    )

if __name__ == "__main__":
    main()

Artifacts saved under -> /content/artifacts/payment


### Model Performance Metrics Dashboard
This cell loads the model metrics from the generated JSON artifacts and displays them in a clean, readable tabular format inside the notebook.

In [18]:
from __future__ import annotations

import json
from pathlib import Path
import pandas as pd

ARTIFACTS_DIR = Path("/content/artifacts")

def load_metrics(model_name):
    path = ARTIFACTS_DIR / model_name / "training_metrics.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

def load_comparison(model_name):
    metrics = load_metrics(model_name)
    if metrics and "algorithm_comparison" in metrics:
        return metrics["algorithm_comparison"]
    return None

# ---------------------------------------------------------
# Print Comparison Tables for All 4 Tasks (5 Models Each)
# ---------------------------------------------------------

tasks = {
    "spending": ("Spending Prediction", ["MAE", "RMSE", "R2"]),
    "savings": ("Savings Prediction", ["MAE", "RMSE", "R2"]),
    "wellness": ("Wellness Prediction", ["MAE", "RMSE", "R2"]),
    "risk": ("Risk Prediction", ["Accuracy", "Precision_weighted", "Recall_weighted", "F1_weighted", "ROC_AUC"]),
}

for model_key, (title, metrics_names) in tasks.items():
    comp = load_comparison(model_key)
    if not comp:
        continue
    print(f"\n{'='*80}")
    print(f"Model: {title.upper()} (5 Models)")
    print(f"{'='*80}")
    
    rows = []
    for algo, vals in comp.items():
        row = {"Algorithm": algo}
        for m in metrics_names:
            key = f"test_{m}" if f"test_{m}" in vals else f"val_{m}" if f"val_{m}" in vals else m
            row[m] = vals.get(key, vals.get(m, "N/A"))
        rows.append(row)
    
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))

# Also print the detailed metrics for each model
print("\n\n" + "="*80)
print("DETAILED TEST METRICS PER MODEL")
print("="*80)

for model_key in ["spending", "savings", "wellness", "risk"]:
    metrics = load_metrics(model_key)
    if not metrics:
        continue
    print(f"\n--- {model_key.upper()} ---")
    if "test" in metrics:
        test_m = metrics["test"]
        if isinstance(test_m, dict):
            for k, v in test_m.items():
                if k != "Confusion_Matrix":
                    print(f"  {k}: {v}")
    if "test_dl" in metrics:
        print("  Deep Learning:")
        for dl_name, dl_m in metrics["test_dl"].items():
            for k, v in dl_m.items():
                print(f"    {dl_name} {k}: {v}")



Model: SPENDING (Selected: RandomForestRegressor)
Target: future_expense - Predicted future-month aggregate spending.

--- Final Performance Evaluation ---


,val,test
MAE,21358.0580,21466.5650
RMSE,83201.9120,81832.5970
R2,-0.0538,-0.0621



--- Algorithm Tuning Comparison (Validation Set) ---


,val_MAE_log,final_params
Algorithm,,
RandomForestRegressor,1.532,"{'n_estimators': 100, 'max_depth': 10, 'min_sa..."
GradientBoostingRegressor,1.562,"{'n_estimators': 100, 'learning_rate': 0.05, '..."
XGBRegressor,1.544,"{'n_estimators': 100, 'learning_rate': 0.05, '..."



Model: SAVINGS (Selected: XGBRegressor)
Target: future_savings - Predicted future-month aggregate savings (aggregate income - aggregate expense).

--- Final Performance Evaluation ---


...

,val_ROC_AUC,final_params
Algorithm,,
LogisticRegression,0.941655,"{'C': 10.0, 'class_weight': 'balanced', 'max_i..."
RandomForestClassifier,0.983471,"{'n_estimators': 250, 'max_depth': 10, 'min_sa..."
XGBClassifier,0.969162,"{'n_estimators': 200, 'learning_rate': 0.1, 'm..."



Model: PAYMENT (Selected: XGBClassifier)
Target: payment_mode - Best payment mode behavior label based on user characteristics.

--- Final Performance Evaluation ---


,val,test
F1_weighted,0.264,0.2273
Accuracy,NaN,0.2344
Precision_weighted,NaN,0.2339
Recall_weighted,NaN,0.2344



--- Algorithm Tuning Comparison (Validation Set) ---


,val_F1,final_params
Algorithm,,
XGBClassifier,0.2640,"{'n_estimators': 200, 'learning_rate': 0.1, 'm..."
RandomForestClassifier,0.2620,"{'n_estimators': 100, 'max_depth': 10, 'min_sa..."
LogisticRegression,0.2599,"{'C': 1.0, 'max_iter': 2000, 'solver': 'lbfgs'}"
